# SystemVerilog Reference Sheet

## 1. Modules

### What are modules?

In an FPGA, we write code to describe hardware circuits. Quartus then turns this code into real digital hardware on the board.

The way we build systems is by creating small blocks (modules) where each block does one specific job, and then connecting them together.

For example, in this project:

- One module converts signals from time-domain to the frequency-domain (FFT)
- One module sends data to the HDMI port
- You will create modules that apply audio effects (echo, robotic voice effect, etc.)

### How to think about modules:
A module is similar to a function (or method) in Python, Java, or C/C++:

- It has inputs
- It has outputs
- It performs a specific task

Key difference:
- A function runs when called
- A module is always running (because it is actually hardware)

### What a module looks like:

Here's an example of a module that takes two numbers as input and outputs their sum.

```systemverilog
module add_two_numbers #(
    // constants go here
    parameter WIDTH = 16
)(
    // inputs and outputs go here
    input logic signed [WIDTH-1:0] x,
    input logic signed [WIDTH-1:0] y,
    output logic signed [WIDTH-1:0] sum
);

    assign sum = x + y;

endmodule
```

### How modules connect together:
To use a module inside another module, we **instantiate** it (plug it in and connect its signals).

This is similar to calling a method in a program, but instead of running code, you are creating hardware and wiring it into your system.

Here’s an example of a calculator module that uses the `add_two_numbers` module from earlier:

```systemverilog
module calculator (
    input  logic signed [15:0] num1,
    input  logic signed [15:0] num2,
    output logic signed [15:0] result
);

    // Instantiate the addition module
    add_two_numbers #(
        .WIDTH(16)
    ) adder (
        .x(num1),
        .y(num2),
        .sum(result)
    );

endmodule
```

#### Instantiating multiple copies

You can instantiate multiple copies of the same module. Each instance must have a unique name.

Here’s an example using two instances of the add_two_numbers module:

```systemverilog
module add_three_numbers (
    input  logic signed [15:0] num1,
    input  logic signed [15:0] num2,
    input  logic signed [15:0] num3,
    output logic signed [15:0] result
);

    logic signed [15:0] sum1, sum2;

    // sum1 = num1 + num2
    add_two_numbers #(
        .WIDTH(16)
    ) adder_1 (
        .x(num1),
        .y(num2),
        .sum(sum1)
    );

    // sum2 = sum1 + num3 = num1 + num2 + num3
    add_two_numbers #(
        .WIDTH(16)
    ) adder_2 (
        .x(sum1),
        .y(num3),
        .sum(sum2)
    );

    assign result = sum2;

endmodule
```

In this workshop, you will write your own module (like `echo.sv`) and then
instantiate it inside `audio_processing.sv`.

## 2. Procedural Blocks

Procedural blocks contain code that is executed sequentially (line-by-line). 

Anything outside of these blocks (like `assign` statements) runs continuously in parallel.

Different types of procedural blocks are used depending on how often the code should run.

### Initial Blocks
Executes only once at the start.
```systemverilog
initial begin
    // runs once at the start
end
```

### Always Block
Executes every time a specified event occurs.

Here is an example of an `always` block:
```systemverilog
assign y = a + b; // always active

always @(posedge clk) begin
    x <= y; // only updates on the rising edge of the clock
end
```

`@(posedge clk)` means it runs on every rising edge of the clock.


## 3. Variables and Arrays

### Variables

Here are some examples of variables in SystemVerilog:

```systemverilog
logic a;                  // holds 1 bit (0 or 1)
logic [7:0] b;            // holds 8 bits (0 to 255)
logic signed [7:0] c;     // holds 8 bits and can be negative (-128 to 127)
```

#### How many bits do you need 
The number of bits determines how large a value you can store.

With $N$ bits, you can you can represent values from $0$ to $2^{N} - 1$

To store the number 25:
- 4 bits: too small (max number it can hold is 15)
- 5 bits: just right (max number it can hold is 31)

### Array

Here are an example of an array in SystemVerilog:

```systemverilog
logic [15:0] arr [0:7];
```

This is means:
- The array that has 8 elements
- Each element is 16 bits wide


### How to Assign Values

#### Blocking and Non-Blocking Statments

Used inside of `always` blocks.

Here's an example of blocking and non-blocking statments:
```systemverilog
always @ (posedge clk) begin
    // blocking 
    y = 9 + 8; (updates immediately)

    // non-blocking 
    x <= y + 2; (updates at end of clock cycle)
end
```
With `<=`, all assignments use the old values before the clock edge.

**Important:** Please use non-blocking (`<=`) by default inside sequential (`always`) blocks.

#### Continuous Assignment

Used outside of `always` blocks.

Here's an example of continuous assignment:
```systemverilog
assign x = a + b;
```

## 4. Conditional Statements

Used inside `always` blocks.

### If-Else Statement

General Syntax:
```systemverilog
if (<condition>) begin
    // statement 1
end 
else begin
	// statement 2
end
```
Here is an example of an if-else statement:
```systemverilog
if (a == 1) begin
    result <= a + b;
end 
else if (a == 2) begin
    result <= a - b;
end
else begin
    result <= a * b;
end
```

### Case Statement 

Used when you have many conditions based on one variable. Often cleaner than multiple `else if` statements.

General Syntax:
```systemverilog
case (<expression>)
   value1: statement1;
   value2: statement2;
   default: statementN;
endcase
```

Here is an example of a case statement:
```systemverilog
case (a)
    1: result <= a + b;
    2: result <= a - b;
    3: result <= a * b;
    default: result <= 0;
endcase```
